####Slowly Changing Dimension - Initial and Incremental

In [0]:

from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

In [0]:
df = spark.read.format("csv")\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load("/Volumes/pyspark/source/rawcsv/products_dim_table.csv")
df.display()

In [0]:
df = df.select("p_id", "p_name", "p_category").filter(col("p_id").isNotNull())
df.display()

In [0]:
initial_run = 0

In [0]:

if(initial_run == 0):
    df.write.format("delta")\
    .mode("append")\
        .saveAsTable("pyspark.source.productsDim")
    initial_run = initial_run + 1
else:
    delta_table = DeltaTable.forName(spark,"pyspark.source.productsDim")
    
    delta_table.alias("tgt").merge(df.alias("src"), "tgt.p_id = src.p_id")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
            .execute()




In [0]:
%sql
select * from pyspark.source.productsdim